# 05 Modelling

This notebook corresponds to the **Machine Learning Modelling** stage.

Goals:
- Predict band of utilisation of bycicles per station in the next 30 minutes


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


PROJECT_ROOT = Path('..').resolve()
FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dublin_bikes_features.parquet'
if not FEATURES_PATH.exists():
    FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dublin_bikes_features.csv.gz'

if FEATURES_PATH.suffix == '.parquet':
    df = pd.read_parquet(FEATURES_PATH).copy()
else:
    df = pd.read_csv(FEATURES_PATH).copy()

df = df.sort_values(['station_id', 'last_reported'])
print(df.shape)
df.head()

(13516758, 13)


,station_id,last_reported,hour_slot,days_of_week,is_weekday,roll_mean_30min,roll_std_30min,utilisation,dock_ratio,empty_risk,full_risk,center_index,area_type_str
0,1,2022-04-27 13:00:02,13,2,True,0.0,0.0,0.0,0.870968,True,False,1,retail
1,1,2022-04-27 13:30:02,13,2,True,0.0,0.0,0.0,0.870968,True,False,1,retail
2,1,2022-04-27 14:00:02,14,2,True,0.0,0.0,0.0,0.870968,True,False,1,retail
3,1,2022-04-27 14:30:02,14,2,True,0.0,0.0,0.0,0.000000,True,False,1,retail
4,1,2022-04-27 15:00:02,15,2,True,0.0,0.0,0.0,0.000000,True,False,1,retail


In [ ]:
# Values for 1 row later
df['utilisation_tplus'] = df.groupby('station_id')['utilisation'].shift(-1)

# Drop rows where we don't know the future
df = df.dropna(subset=['utilisation_tplus']).copy()

# Define bands for 30‑min ahead utilisation 
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
band_labels = ["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"]

df['util_band_30min'] = pd.cut(
    df['utilisation_tplus'].clip(0.0, 1.0),
    bins=bins,
    labels=band_labels,
    include_lowest=True
)

In [3]:
# Features & Time based split

feature_cols = [
    'hour_slot', 'days_of_week', 'is_weekday', 
    'roll_std_30min','utilisation',
    'center_index', 'area_type_str'
]

X = df[feature_cols]
y = df['util_band_30min'].copy()          

# One‑hot encode area type
X = pd.get_dummies(X, columns=['area_type_str'], drop_first=True)

# Time‑based split: last 120 days as test
cutoff = df['last_reported'].max() - pd.Timedelta(days=120)
train_mask = df['last_reported'] < cutoff

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
y_train.value_counts(normalize=True).sort_index()

Train shape: (11091558, 8) Test shape: (2425084, 8)


util_band_30min
0–20%      0.348091
20–40%     0.215165
40–60%     0.172486
60–80%     0.125344
80–100%    0.138913
Name: proportion, dtype: float64

In [4]:
# Normalization & Training the Multinomial Logistic Model

scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(
    max_iter=300,
    class_weight='balanced',
    solver='lbfgs',
)

log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)

print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=band_labels))

              precision    recall  f1-score   support

       0–20%       0.98      0.95      0.97    839634
      20–40%       0.89      0.91      0.90    516856
      40–60%       0.90      0.90      0.90    423475
      60–80%       0.89      0.90      0.90    311529
     80–100%       0.95      0.96      0.96    333590

    accuracy                           0.93   2425084
   macro avg       0.92      0.93      0.92   2425084
weighted avg       0.93      0.93      0.93   2425084

Confusion matrix:
 [[801334  37739    393    115     53]
 [ 18997 471497  25720    464    178]
 [   565  18455 381476  22276    703]
 [    53    266  14862 281418  14930]
 [    24     24    104  12623 320815]]


In [5]:
# To compare true bands to predicted bands
test_results = df.loc[~train_mask, ['station_id', 'last_reported']].copy()
test_results['true_band_30min'] = y_test.values
test_results['pred_band_30min'] = y_pred

proba_cols = [f"p_{b}" for b in band_labels]
test_results[proba_cols] = y_proba

test_results.head()

,station_id,last_reported,true_band_30min,pred_band_30min,p_0–20%,p_20–40%,p_40–60%,p_60–80%,p_80–100%
96500,1,2025-05-03 23:55:00,40–60%,40–60%,0.000001,0.050713,0.875032,0.074247,0.000007
96501,1,2025-05-04 00:05:00,40–60%,40–60%,0.000001,0.055106,0.879406,0.065481,0.000006
96502,1,2025-05-04 00:15:00,40–60%,40–60%,0.000001,0.055106,0.879406,0.065481,0.000006
96503,1,2025-05-04 00:25:00,40–60%,40–60%,0.000001,0.055106,0.879406,0.065481,0.000006
96504,1,2025-05-04 00:35:00,40–60%,40–60%,0.000001,0.055106,0.879406,0.065481,0.000006


In [6]:
# Function to predict the Band of availability of the stations at a specific time

def predict_30min_all_stations(time):
    # 1. Convert input time to pandas datetime
    time = pd.to_datetime(time)

    # 2. Subset the data: Take all rows in df before or equal to the given 'time'
    sub = df[df['last_reported'] <= time].copy()
    if sub.empty:
        raise ValueError("No data before this time.")

    # 3. Among those, keep only the latest measurement per station
    #    - Sort by 'station_id' and 'last_reported'
    #    - For each station_id, keep the last (most recent) row
    sub = (
        sub.sort_values(['station_id', 'last_reported'])
           .groupby('station_id')
           .tail(1)
    )

    # 4. Prepare input features for the model:
    #    - Select only the columns used in training (feature_cols)
    #    - Convert categorical variables to dummies (matching model expectations)
    X_rows = sub[feature_cols].copy()
    X_rows = pd.get_dummies(X_rows, columns=['area_type_str'], drop_first=True)

    # 5. Ensure feature columns match those used in training:
    #    - For each column in the training set, add if missing (set to 0)
    #    - Rearrange columns to match the training set
    for col in X_train.columns:
        if col not in X_rows.columns:
            X_rows[col] = 0
    X_rows = X_rows[X_train.columns]

    # 6. Transform the data (normalization/scaling) as during training
    X_rows_scaled = scaler.transform(X_rows)

    # 7. Run the model to get predictions
    y_pred_rows = log_reg.predict(X_rows_scaled)            # predicted band class
    y_proba_rows = log_reg.predict_proba(X_rows_scaled)     # probability for each band

    # 8. Build the output DataFrame:
    #    - Copy station_id and last_reported columns
    #    - Add predicted band
    out = sub[['station_id', 'last_reported']].copy()
    out['pred_band_30min'] = y_pred_rows

    #    - Add predicted probability for each band (probability columns)
    for i, band in enumerate(band_labels):
        out[f"p_{band}"] = y_proba_rows[:, i]

    # 9. Return results sorted by station, and reset index for a clean DataFrame
    return out.sort_values('station_id').reset_index(drop=True)

In [7]:
preds = predict_30min_all_stations("2024-08-15 08:30:00")
preds.head()

,station_id,last_reported,pred_band_30min,p_0–20%,p_20–40%,p_40–60%,p_60–80%,p_80–100%
0,1,2024-08-15 08:30:00,20–40%,1.516298e-02,0.911888,7.292024e-02,2.860247e-05,7.401640e-12
1,2,2024-08-15 08:25:00,0–20%,9.977114e-01,0.002289,5.840064e-08,7.249480e-15,3.043341e-25
2,3,2024-08-15 08:30:00,0–20%,7.003118e-01,0.299174,5.138983e-04,4.193532e-09,1.800002e-17
3,4,2024-08-15 08:30:00,60–80%,3.540892e-10,0.000802,3.519318e-01,6.449965e-01,2.270161e-03
4,5,2024-08-15 08:30:00,0–20%,5.224679e-01,0.475884,1.647766e-03,2.742375e-08,2.809349e-16


In [9]:
df[df['last_reported'] == "2024-08-15 09:00:00"]

,station_id,last_reported,hour_slot,days_of_week,is_weekday,roll_mean_30min,roll_std_30min,utilisation,dock_ratio,empty_risk,full_risk,center_index,area_type_str,utilisation_tplus,util_band_30min
52173,1,2024-08-15 09:00:00,9,3,True,0.266129,0.013968,0.258065,0.741935,False,False,1,retail,0.290323,20–40%
174524,2,2024-08-15 09:00:00,9,3,True,0.000000,0.000000,0.000000,1.000000,True,False,2,residential,0.000000,0–20%
290796,3,2024-08-15 09:00:00,9,3,True,0.083333,0.047140,0.050000,0.950000,False,False,1,residential,0.100000,0–20%
409579,4,2024-08-15 09:00:00,9,3,True,0.600000,0.000000,0.600000,0.400000,False,False,2,residential,0.500000,40–60%
528619,5,2024-08-15 09:00:00,9,3,True,0.130000,0.018708,0.100000,0.900000,False,False,2,residential,0.100000,0–20%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12869974,112,2024-08-15 09:00:00,9,3,True,0.566667,0.043033,0.500000,0.500000,False,False,2,residential,0.466667,40–60%
12990854,113,2024-08-15 09:00:00,9,3,True,0.100000,0.000000,0.100000,0.900000,False,False,2,residential,0.100000,0–20%
13103281,114,2024-08-15 09:00:00,9,3,True,0.937500,0.021651,0.950000,0.000000,False,False,3,work,0.950000,80–100%
13223383,115,2024-08-15 09:00:00,9,3,True,0.508333,0.014434,0.500000,0.500000,False,False,2,residential,0.466667,40–60%
